In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

tables = {
    "PROD_HARMONIZED.PRODUCT.TENANT_PROF": {
        "account_col": "ACCOUNT_ID",
        "filter": "CRNT_RCRD_IND = 'Y'"
    },
    "PROD_HARMONIZED.SALES.VW_ACCOUNT_PROF": {
        "account_col": "ACCOUNT_ID",
        "filter": "CRNT_RCRD_IND = 'Y'"
    },
    "PROD_PRESENTATION.PROFESSIONAL_SERVICES.VW_PS_PROJECTS": {
        "account_col": "ACCOUNT_ID",
        "filter": "CRNT_RCRD_IND = 'Y'"
    }
}

queries = []

for table, config in tables.items():
    account_col = config["account_col"]
    filter_clause = config["filter"]

    where_clause = f"WHERE {filter_clause}" if filter_clause else ""

    queries.append(f"""
        SELECT
            $$ {table} $$ AS table_name,
            $$ {account_col} $$ AS account_id_column,
            $$ {filter_clause} $$ AS filter_applied,
            COUNT(*) AS total_row_count,
            COUNT_IF({account_col} IS NULL) AS null_account_id_count
        FROM {table}
        {where_clause}
    """)

final_sql = "\nUNION ALL\n".join(queries)

results = session.sql(final_sql).collect()

for row in results:
    print(
        row["TABLE_NAME"],
        row["ACCOUNT_ID_COLUMN"],
        row["FILTER_APPLIED"],
        row["TOTAL_ROW_COUNT"],
        row["NULL_ACCOUNT_ID_COUNT"]
    )